# 10-2절 연습 문제 풀이

이 노트북은 10-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch10/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 10-2/10-3절 공통 - 인코더/디코더 전용 트랜스포머
import math, random as _r
from torch.utils.data import Dataset, DataLoader
PAD, SOS, EOS, UNK, CLS = '<pad>', '<sos>', '<eos>', '<unk>', '[CLS]'

class DateValidator(nn.Module):
    """인코더만 사용하는 트랜스포머 (BERT 계열)"""
    def __init__(self, vocab, d_model=128, nhead=4, layers=2, pooling='cls'):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model, padding_idx=0)
        self.pos = nn.Embedding(64, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4,
                                               batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, layers)
        self.fc = nn.Linear(d_model, 2)
        self.pooling = pooling
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.encoder(self.emb(x) + self.pos(pos),
                         src_key_padding_mask=(x == 0))
        if self.pooling == 'cls':
            pooled = h[:, 0]                       # [CLS] 토큰 위치
        else:
            mask = (x != 0).unsqueeze(-1).float()  # <pad> 제외 평균
            pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(pooled)

class OzWriterTransformer(nn.Module):
    """디코더만 사용하는 트랜스포머 (GPT 계열)"""
    def __init__(self, vocab, d_model=256, nhead=4, layers=4, max_len=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4,
                                           batch_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers)
        self.fc = nn.Linear(d_model, vocab)
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        mask = nn.Transformer.generate_square_subsequent_mask(x.size(1),
                                                             device=x.device)
        h = self.blocks(self.emb(x) + self.pos(pos), mask=mask, is_causal=True)
        return self.fc(h)

## 연습 10-6

[표 10-7]을 살펴보면 모델이 2월 29일이 실제로 존재하는지 여부를 제대로 판정하지 못한다는 사실을 알 수 있다. 이 문제를 해결하려면 두 가지 접근 방법이 가능하다.

데이터셋을 보완해 윤년의 2월 29일도 잘 분류하는 모델 만들기

그 해가 윤년인지 아닌지만 판정하는 모델을 따로 만들어, 기존 모델과 함께 사용하기

두 방법을 모두 시도해 본 후 어떤 방법이 더 적절한지 설명해 보자.

### 풀이

**접근 1) 데이터셋 보완**
윤년 판정은 규칙이 명확하다(4의 배수이면서 100의 배수가 아니거나, 400의 배수). 학습 데이터에 2월 29일 사례(윤년의 유효한 2/29와 평년의 무효한 2/29)를 **의도적으로 균형 있게 포함**하면 모델이 연도와 날짜의 관계를 배울 수 있다.

**접근 2) 규칙 기반 후처리**
모델 출력에 더해 파이썬 `datetime`으로 실제 날짜인지 검사해 최종 판정을 보정한다.

**어느 쪽이 나은가**: 윤년 규칙은 **결정적이고 예외가 없는 규칙**이라 딥러닝으로 배우게 할 이유가 약하다. 모델은 데이터 분포를 근사할 뿐이라 400년 주기의 예외(1900년은 평년, 2000년은 윤년)까지 배우려면 매우 많은 데이터가 필요하다. **규칙으로 풀 수 있는 것은 규칙으로 푸는 편이 정확하고 저렴하다**. 실무에서도 모델과 규칙을 함께 쓰는 하이브리드가 일반적이다.

In [ ]:
from datetime import date
def is_valid_date(y, m, d):
    try:
        date(y, m, d); return True
    except ValueError:
        return False
for y in (1900, 2000, 2023, 2024):
    print(f'{y}-02-29 유효 여부: {is_valid_date(y, 2, 29)}')

## 연습 10-7

[CLS] 토큰 대신 입력 토큰 전체의 출력 벡터를 평균 풀링한 결과를 분류기에 입력하도록 DateValidator 클래스를 수정해 보자. 같은 데이터셋과 같은 하이퍼파라미터로 학습해 [CLS] 토큰 방식과 성능을 비교하고, 두 방식의 장단점도 정리해 보자.

In [ ]:
# [CLS] 방식과 평균 풀링 방식 비교 (구조 확인용 더미 데이터)
vocab = {PAD: 0, CLS: 1, **{str(d): i + 2 for i, d in enumerate(range(10))}}
x = torch.randint(2, len(vocab), (4, 16)); x[:, 0] = vocab[CLS]
for pooling in ('cls', 'mean'):
    m = DateValidator(len(vocab), pooling=pooling)
    print(f'{pooling:5s} 풀링 -> 출력 {tuple(m(x).shape)}')

**[CLS] 방식**은 문장 전체 정보를 모으라고 **전용 자리 하나를 배정**하는 방식이다. 어텐션이 그 위치에 필요한 정보를 모으도록 학습된다.

**평균 풀링**은 모든 토큰의 출력을 평균낸다. 별도 학습 없이도 안정적이며 짧은 입력에서는 [CLS]보다 나을 때도 있다. 다만 `<pad>` 위치를 반드시 제외해야 한다(위 코드의 `mask`).

일반적으로 데이터가 많으면 [CLS]가, 적으면 평균 풀링이 유리한 경향이 있다.

## 연습 10-8

6장 오즈의 띄어쓰기 모델(모델 8)을 순환 신경망 대신 인코더만 사용하는 트랜스포머로 만들어 보자.

In [ ]:
# 6장 띄어쓰기 모델을 인코더 전용 트랜스포머로
class SpacingTransformer(nn.Module):
    def __init__(self, vocab, d_model=128, nhead=4, layers=2, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model, padding_idx=0)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4,
                                           batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers)
        self.fc = nn.Linear(d_model, 1)          # 위치마다 띄어쓰기 여부
    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.encoder(self.emb(x) + self.pos(pos), src_key_padding_mask=(x == 0))
        return self.fc(h).squeeze(-1)

m = SpacingTransformer(100)
x = torch.randint(1, 100, (4, 32))
print(f'입력 {tuple(x.shape)} -> 출력 {tuple(m(x).shape)} (위치별 로짓)')

띄어쓰기는 **각 위치마다 분류**하는 과제이므로 [CLS] 같은 요약 토큰이 필요 없다. 인코더 출력을 그대로 위치별 분류기에 넣는다.

양방향 LSTM과 달리 트랜스포머는 처음부터 **모든 위치가 서로를 참조**하므로 별도 처리 없이 앞뒤 문맥을 함께 본다. 인과 마스크를 쓰지 않는 것이 핵심이다.

## 연습 10-9

[도전 문제] 10-1절의 날짜 변환기 트랜스포머 모델의 예측 함수([코드 10-5]와 [코드 10-6])를 생성 결과와 생성 신뢰도를 함께 반환하도록 수정해 보자. 각 토큰별 신뢰도를 조합하되, 생성 결과의 길이에 무관하게 신뢰도를 계산하는 방법을 고안해 적용해 보자.

In [ ]:
@torch.no_grad()
def generate_with_confidence(model, src, tv, rev, max_len=16):
    """생성 결과와 길이로 정규화한 신뢰도를 함께 반환"""
    model.eval()
    tgt = torch.tensor([[tv[SOS]]], device=src.device)
    log_probs, tokens = [], []
    for _ in range(max_len):
        logits = model(src, tgt)[:, -1]
        probs = torch.softmax(logits, dim=-1)
        p, nxt = probs.max(dim=-1)
        if nxt.item() == tv[EOS]: break
        log_probs.append(torch.log(p).item()); tokens.append(rev[nxt.item()])
        tgt = torch.cat([tgt, nxt.unsqueeze(0)], dim=1)
    # 길이로 정규화: 평균 로그 확률 -> 지수 변환
    conf = math.exp(sum(log_probs) / len(log_probs)) if log_probs else 0.0
    return ''.join(tokens), conf

print('생성 신뢰도는 토큰별 확률의 로그 평균을 지수로 되돌려 계산한다.')
print('길이로 나누지 않으면 긴 문자열일수록 확률 곱이 작아져 불리해진다.')

토큰별 확률을 그대로 곱하면 **길이가 길수록 값이 작아져** 짧은 출력이 항상 유리해진다. 로그 확률의 **평균**을 구한 뒤 지수로 되돌리면 길이에 공정한 신뢰도가 된다. 빔 서치의 길이 정규화와 같은 원리다.

## 연습 10-10

[도전 문제] 정렬 시리즈의 네 번째 문제는 주어진 수열이 정렬돼 있는지 판정하는 문제다. 1에서 1,000 사이의 정수 10개를 쉼표로 구분한 문자열(예) 5, 17, 200, 250, ...)을 입력받아, 숫자가 오름차순인지, 내림차순인지, 아니면 정렬되어 있지 않은지를 판단하는 인코더만 사용하는 트랜스포머 모델을 만들어 보자.

In [ ]:
# 수열이 정렬돼 있는지 판정하는 분류 모델
def make_sorted_pairs(n=5000, count=10, seed=SEED):
    rng = _r.Random(seed); data = []
    for _ in range(n):
        nums = sorted(rng.randint(1, 1000) for _ in range(count))
        if rng.random() < 0.5:                    # 절반은 순서를 흐트러뜨린다
            i, j = rng.randrange(count), rng.randrange(count)
            nums[i], nums[j] = nums[j], nums[i]
        label = int(all(a <= b for a, b in zip(nums, nums[1:])))
        data.append((', '.join(map(str, nums)), label))
    return data

data = make_sorted_pairs()
vocab = {PAD: 0, CLS: 1, UNK: 2}
for c in sorted({c for s, _ in data for c in s}): vocab.setdefault(c, len(vocab))
def encode(s, L=60):
    ids = [vocab[CLS]] + [vocab.get(c, vocab[UNK]) for c in s][:L - 1]
    return ids + [0] * (L - len(ids))
X = torch.tensor([encode(s) for s, _ in data])
Y = torch.tensor([y for _, y in data])
print(f'정렬된 수열 비율 {Y.float().mean():.2%}')

torch.manual_seed(SEED)
model = DateValidator(len(vocab)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
loader = DataLoader(list(zip(X, Y)), batch_size=64, shuffle=True)
for e in range(1, 11):
    model.train(); c = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x); loss = crit(out, y)
        opt.zero_grad(); loss.backward(); opt.step()
        c += (out.argmax(1) == y).sum().item(); n += len(y)
    print(f'{e}/10 정확도 {c / n * 100:.2f}%')

정렬 여부 판정은 **인접한 두 수를 비교**하는 국소 관계만 보면 되므로 인코더 전용 트랜스포머에 잘 맞는다. 다만 숫자를 문자 단위로 다루므로 '200'과 '30'의 크기 비교를 자릿수까지 고려해 배워야 해서 생각보다 까다롭다.

## 연습 10-11

[도전 문제] 어휘 사전을 글자 단위가 아니라 월 이름, 구분자, 두 자리 숫자, 네 자리 숫자 같은 미리 정한 단위 토큰의 집합으로 바꾸고, 입력 문자열을 그것에 맞게 토큰화하는 전처리 함수를 작성해 모델을 다시 학습해 보자. 결과를 토대로 토큰화 단위가 모델 성능에 어떤 영향을 미쳤는지 분석해 보자.

In [ ]:
import re
def tokenize_date(text):
    """월 이름, 구분자, 두 자리/네 자리 숫자 단위로 토큰화"""
    MONTHS = ['January','February','March','April','May','June','July','August',
              'September','October','November','December']
    pattern = '|'.join([*[m for m in MONTHS], *[m[:3] for m in MONTHS],
                        r'\d{4}', r'\d{1,2}', r'[-/,]', r'\s+'])
    return [t for t in re.findall(pattern, text) if not t.isspace()]

for s in ['01 February 2026', 'Feb 01, 2026', '2026-02-01']:
    print(f'{s:20s} -> {tokenize_date(s)}')

단위 토큰화를 쓰면 시퀀스 길이가 크게 줄어(문자 16개 → 토큰 3~5개) 어텐션 계산이 가벼워지고, '2026'이 하나의 의미 단위로 다뤄져 학습이 쉬워진다.

대신 어휘 사전이 커지고, 정의하지 않은 형식은 처리하지 못한다. 12장 LLM의 서브워드 토크나이저가 이 둘을 절충한 방식이다.